In [5]:
import sys, pandas as pd, numpy as np
sys.path.insert(0, "..")
from config import *

In [6]:
pairs = pd.read_parquet(ROOT / "data" / "processed_traded_only" / "pairs.parquet",
                        columns=["split", "type", "delta", "dV", "dS", "y", "w"])
pairs.groupby(["split", "type"]).size()

split  type
train  C       330452
       P       445631
val    C       192599
       P       216424
test   C       369936
       P       400668
dtype: int64

In [7]:
# Hull-White one-number correction: delta* = lambda * delta
# lambda = sum(w * delta * y) / sum(w * delta^2), fitted on the training years only
train = pairs[pairs["split"] == "train"]

lam = {}
for t in ["C", "P"]:
    d = train[train["type"] == t]
    lam[t] = (d["w"] * d["delta"] * d["y"]).sum() / (d["w"] * d["delta"] ** 2).sum()

print("lambda calls:", round(lam["C"], 4))
print("lambda puts: ", round(lam["P"], 4))

lambda calls: 0.9385
lambda puts:  1.0493


In [8]:
# gain = 1 - (error with lambda*delta) / (error with plain delta); positive = better than Black-Scholes
for s in ["val", "test"]:
    for t in ["C", "P"]:
        d = pairs[(pairs["split"] == s) & (pairs["type"] == t)]
        err_bs = ((d["dV"] - d["delta"] * d["dS"]) ** 2).sum()
        err_hw = ((d["dV"] - lam[t] * d["delta"] * d["dS"]) ** 2).sum()
        print(f"{s:5s} {t}: gain = {1 - err_hw / err_bs:.4f}")

val   C: gain = -0.0032
val   P: gain = -0.0506
test  C: gain = 0.1588
test  P: gain = 0.0457


### What this shows

Hull & White's simplest correction scales the Black-Scholes delta by one number fitted on 2016–2018: λ = 0.94 for calls, 1.05 for puts — the same directions Hull & White (2017) found (0.91 and 1.08), so the data carries the effect the project studies.

The gain depends on the year. In 2020 the correction reduces hedging error by 15.9% (calls) and 4.6% (puts). In the calm year 2019 it slightly increases it (−0.3% and −5.1%). A fixed correction helps when volatility spikes and hurts when it does not; this is the motivation for models that condition on the option's features, which the next notebooks test.